# Unit 3 — Reference Solution: Follow a Bus Line as a Corridor, Then Forecast Each Segment

> ## Disclaimer — AI-generated reference, provided for learning
>
>> **© 2026 Ben Galon. All rights reserved.** Part of the **Geo-AI course (The Arena)** — provided to enrolled students for course use, not for redistribution.
>
> **This solution notebook was generated by the course's AI assistant** (with
> full access to the unit's research synthesis, the Unit 2 map-matching demo +
> solution, the Unit 3 forecasting demo, and the practice task). It is shared
> with you **from day one, on purpose**: seeing a complete, runnable solution
> shows that the practice task is *feasible* and gives you a concrete reference
> for the *shape* of a strong direct -> interpret -> extend answer.
>
> **Two honest caveats:**
> 1. **It has not been fully Run-All'd end-to-end on Colab.** It passed static
>    checks (valid notebook JSON; every code cell compiles) and the corridor +
>    speed-from-ping-pairs pipeline was numerically spot-checked on a real
>    single-line TLV slice. `statsforecast` and `folium` still need a human
>    **Colab Run-All** for full parity. Treat it as a reference, not a
>    guaranteed-runnable artifact — if a cell breaks, fixing it is good practice.
> 2. **The task is open-ended by design — there is no single correct answer.**
>    This is *one* defensible path (all-TLV pings -> a line-shaped corridor ->
>    ~10 segments -> a per-segment breakeven). Your own corridor line,
>    segmentation strategy, speed filters, and dwell handling can be just as
>    correct. Do not copy this; use it to check your reasoning. You are graded on
>    the quality of your own direct -> interpret -> extend loop, not on matching
>    this notebook.

**The synthesis.** Unit 2 taught you to **map-match** raw bus pings onto the road.
The Unit 3 demo taught you the **breakeven horizon** on one METR-LA sensor. This
notebook joins them: from **every** TLV bus ping we take **one bus line's shape**
as a thin **±10 m corridor** (EPSG:2039), **cut it into ~10 segments**, build a
per-segment speed series **from consecutive ping pairs** (not the `velocity`
column), and run the demo's breakeven workflow on **each segment**.

**Stack:** `shapely` + `pyproj` (EPSG:2039) for the corridor geometry,
`statsmodels` (ADF) + `statsforecast` (fixed-order SARIMA, walk-forward) for the
forecast, `folium` for the maps. **No new heavy dependencies.**

**What it covers:** the baseline (line-shaped corridor -> ~10 segments ->
ping-pair speed -> per-segment SARIMA breakeven vs the weekday/weekend
historical-average + seasonal-naive floors) and the extensions: **(a)** the
upstream->downstream **lead** (the U5 message-passing seed) — *the designed
surprise*; **(b)** a segmentation A/B (10 equal pieces vs fixed 150 m);
**(c)** the `velocity==0` dwell-in-vs-out decision; **(d)** wrong-class reasoning.

## 0. Setup

In [ ]:
# This practice needs ONLY the unit-3 extra. The corridor geometry uses the
# LIGHT geometry libs (shapely + pyproj for EPSG:2039) — now part of the unit-3
# extra — NOT osmnx/geopandas: the corridor follows one bus line's GPS centerline
# buffered to a +-10 m ribbon, and nearest-segment snapping uses shapely's STRtree
# (built into shapely 2.x, so rtree is not needed). The rest is the forecasting stack.
# Locally: assumes `uv sync --extra unit-3`.
import sys
if "google.colab" in sys.modules:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/bgalon/geo-graph-learning/main/setup_colab.py",
        "setup_colab.py",
    )
    from setup_colab import setup_unit
    setup_unit("unit-3")   # geometry (shapely/pyproj) + forecasting (statsmodels/statsforecast/folium)

## Smoke test — fail fast, before any teaching content

In [ ]:
# --- Unit 3 practice smoke test: geometry (shapely/pyproj) + forecast (statsmodels/statsforecast) ---
# The corridor pipeline is shapely + pyproj only; osmnx/geopandas are NOT used.
_smoke_err = None
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    # Corridor-geometry half (light: shapely + pyproj, EPSG:2039)
    import shapely
    from shapely.geometry import LineString, Point
    from shapely.strtree import STRtree
    from pyproj import Transformer

    # Forecasting half (Unit 3 demo stack)
    from statsmodels.tsa.stattools import adfuller
    from statsforecast import StatsForecast
    from statsforecast.models import ARIMA, HistoricAverage, SeasonalNaive

    import folium
    from folium import LayerControl
    import branca.colormap as bcm

    # Projection sanity (TLV is in EPSG:2039, the Israeli TM grid).
    _t = Transformer.from_crs("EPSG:4326", "EPSG:2039", always_xy=True)
    assert abs(_t.transform(34.78, 32.07)[0]) > 0

    # STRtree nearest-segment snapping (shapely 2.x — no rtree dependency).
    _segs = [LineString([(0, 0), (100, 0)]), LineString([(100, 0), (200, 0)])]
    _tree = STRtree(_segs)
    assert int(_tree.nearest(Point(150, 5))) == 1

    # Speed-from-pairs primitive: distance / dt on two metric points.
    import math
    _d = math.hypot(189000 - 188940, 663000 - 662970)  # ~67 m
    assert 60 < _d < 75

    # ADF runs on a trivial series.
    _stat, _p, *_ = adfuller(np.r_[np.sin(np.linspace(0, 20, 200))])
    assert np.isfinite(_p)

    # statsforecast ARIMA constructs (numba warms on first real fit).
    _m = ARIMA(order=(1, 1, 1), seasonal_order=(1, 1, 1), season_length=4)
    assert _m is not None
except Exception as exc:
    _smoke_err = exc

if _smoke_err is not None:
    print("=" * 70)
    print("SMOKE TEST FAILED — environment is not ready. See KNOWN_ISSUES.md")
    print(f"  ({type(_smoke_err).__name__}: {_smoke_err})")
    print("=" * 70)
    raise _smoke_err from None

print("Smoke test passed — geometry + forecast stacks are ready.")

## 1. Data — every TLV bus ping (all 1458 lines)

Same artifact as the Unit 2 practice: `tlv_all.parquet`, ~4.7 M pings over the
TLV bbox. We load it once, then (in the next section) clip to a tiny corridor
**before** any snapping — that drops millions of pings to a few thousand and
keeps Colab fast.

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_ID = "1BqGMOa5urESNf-X3FHlMXZiFwIDp4EqQ"   # public tlv_all.parquet (~48 MB) — SAME id as U2
LOCAL = "tlv_all.parquet"
if not os.path.exists(LOCAL):
    import gdown
    gdown.download(id=DATA_ID, output=LOCAL, quiet=False)

df = pd.read_parquet(LOCAL)
df["recorded_at_time"] = pd.to_datetime(df["recorded_at_time"], utc=True)   # already tz-aware UTC

# TLV bbox (from the data cut, NOT from OSM).
LAT0, LAT1, LON0, LON1 = 32.00, 32.15, 34.74, 34.88
print(f"All-TLV pings: {len(df):,}")
print(f"distinct lines: {df.line_ref.nunique()} · vehicles: {df.vehicle_ref.nunique()}")
print(f"days: {df.recorded_at_time.dt.tz_convert('Asia/Jerusalem').dt.date.nunique()} "
      f"· span {df.recorded_at_time.min()} -> {df.recorded_at_time.max()}")

## 2. Helpers — projection + a basemap helper

The heavy mechanics live in helper functions. You map-matched by hand in U2, so
here you build the corridor from a bus line's **shape** and spend your effort on
the **segmentation design** and the **interpretation**.

First, the projection transformers (everything metric in EPSG:2039) and a folium
basemap helper.

In [ ]:
from pyproj import Transformer

_tf = Transformer.from_crs("EPSG:4326", "EPSG:2039", always_xy=True)   # forward (to metres)
inv = Transformer.from_crs("EPSG:2039", "EPSG:4326", always_xy=True)   # inverse (back to lat/lon)

# Project ALL pings once to metres (EPSG:2039) so every distance/speed is metric.
df["x"], df["y"] = _tf.transform(df.lon.values, df.lat.values)

def base_map(center, zoom=14):
    "Folium map with switchable basemaps (the U2/U3 demo helper)."
    import folium
    m = folium.Map(location=center, zoom_start=zoom, tiles=None, control_scale=True)
    folium.TileLayer("OpenStreetMap", name="OSM").add_to(m)
    folium.TileLayer("CartoDB positron", name="Carto (light)").add_to(m)
    return m

print("Projection ready: _tf (->2039), inv (->4326); all pings carry metric x,y.")

### 2a. Pick a corridor LINE and build its ±10 m shape

Instead of a density blob, the corridor **follows one bus line's actual path**.
We pick a line, recover its centerline from its own GPS, trim the densest ~2 km,
and keep a thin **±10 m ribbon** around it. Every line's pings that fall inside
that ribbon feed the per-segment speeds — the all-TLV payoff.

In [ ]:
import numpy as np, shapely
from shapely.geometry import LineString
from shapely.ops import substring

# Pick ONE bus line whose SHAPE is the corridor (Ben\'s call). Default: auto-pick
# the busiest line through central TLV. Override CORRIDOR_LINE_REF for a specific
# line. NOTE: GTFS is empty here, so a *public* line number (e.g. "5") cannot be
# auto-resolved to its line_ref — set the code directly if you know it.
CORRIDOR_LINE_REF = None        # e.g. "13429"; None => auto-pick busiest central line

CENTRAL = dict(lon_w=34.76, lon_e=34.81, lat_s=32.04, lat_n=32.10)
central = df[df.lon.between(CENTRAL["lon_w"], CENTRAL["lon_e"]) &
            df.lat.between(CENTRAL["lat_s"], CENTRAL["lat_n"])]
if CORRIDOR_LINE_REF is None:
    CORRIDOR_LINE_REF = central.line_ref.value_counts().idxmax()
line = central[central.line_ref == CORRIDOR_LINE_REF].copy()
print(f"Corridor line_ref = {CORRIDOR_LINE_REF}: {len(line):,} central pings, "
      f"{line.vehicle_ref.nunique()} vehicles")

def line_centerline(px, py, n_bins=80, min_per_bin=5):
    """Clean centerline from a line\'s GPS: order pings along the principal axis
    (PCA), then take the MEDIAN position per bin. This averages out the 16 days of
    back-and-forth trips and the GPS jitter into one smooth path on the road."""
    X = np.column_stack([px, py]); c = X.mean(0)
    axis = np.linalg.svd(X - c, full_matrices=False)[2][0]      # 1st principal direction
    s = (X - c) @ axis
    e = np.linspace(s.min(), s.max(), n_bins + 1)
    b = np.clip(np.digitize(s, e) - 1, 0, n_bins - 1)
    cl = [(np.median(px[b == k]), np.median(py[b == k]))
          for k in range(n_bins) if (b == k).sum() >= min_per_bin]
    return LineString(cl).simplify(15)                          # 15 m: drop residual wiggle

route = line_centerline(line.x.values, line.y.values)

# Trim to the densest ~2 km window so the corridor sits where the data is rich.
CORRIDOR_LEN = 2000.0
L = route.length
if L > CORRIDOR_LEN:
    samp = line.sample(min(6000, len(line)), random_state=0)
    d_along = shapely.line_locate_point(route, shapely.points(samp.x.values, samp.y.values))
    centers = np.linspace(CORRIDOR_LEN / 2, L - CORRIDOR_LEN / 2, 40)
    ctr = max(centers, key=lambda c0: int(np.sum(np.abs(d_along - c0) <= CORRIDOR_LEN / 2)))
    a = max(0.0, ctr - CORRIDOR_LEN / 2)
    spine_line = substring(route, a, min(L, a + CORRIDOR_LEN))
else:
    spine_line = route
print(f"Corridor spine: {spine_line.length:.0f} m · simple geometry = {spine_line.is_simple}")

### 3. DESIGN the segmentation — a thin ±10 m ribbon, cut into ~10 segments

The corridor is the **shape of one bus line**, not a density blob: we follow the
line's centerline and keep a **±10 m ribbon** on each side (≈ one carriageway).
Every ping from **every** line inside the ribbon feeds the per-segment speed.

**The graded decision (yours to make):** how to cut the ribbon into segments.
The reference cuts **10 equal ~200 m pieces**; you might cut junction-to-junction,
per OSM edge, or by another fixed length. Name your choice and defend it — the
agent must not pick it for you. Extension (b) re-cuts the same ribbon to show how
much the choice matters.

In [ ]:
import shapely
from shapely.ops import substring
from shapely.strtree import STRtree

# The ±10 m corridor ribbon: keep every ping within 10 m of the line\'s centerline.
HALF_WIDTH = 10.0        # metres each side (Ben: +-10 m, not more) — a config knob
bx0, by0, bx1, by1 = spine_line.bounds
cand = df[(df.x.between(bx0 - HALF_WIDTH, bx1 + HALF_WIDTH)) &
          (df.y.between(by0 - HALF_WIDTH, by1 + HALF_WIDTH))].copy()
cand["d_spine"] = shapely.distance(shapely.points(cand.x.values, cand.y.values), spine_line)
corr = cand[cand.d_spine <= HALF_WIDTH].copy()
print(f"Pings within +-{HALF_WIDTH:.0f} m of the line: {len(corr):,} "
      f"(pooled from {corr.line_ref.nunique()} lines, {corr.vehicle_ref.nunique()} vehicles)")

# Cut the spine into ~10 segments (index increases downstream).
N_SEG = 10
def cut_segments(line, n):
    L = line.length
    return [substring(line, i * L / n, (i + 1) * L / n) for i in range(n)]
segments = cut_segments(spine_line, N_SEG)
print(f"{N_SEG} segments, lengths (m): {[round(s.length) for s in segments]}")

# Coverage read: how many pings land on each segment (nearest-segment snap).
_cov_tree = STRtree(segments)
corr["seg"] = _cov_tree.nearest(shapely.points(corr.x.values, corr.y.values))
print("pings per segment:", corr.seg.value_counts().sort_index().to_dict())

## 4. Snap pings to segments + ping-pair speed (the U2 + U3 join)

Now the core of the task, in two helpers:

1. **`attribute_segment`** — for each ping, the **nearest corridor segment**
   (point-to-line distance, metric). This is the matching step; with a thin
   ±10 m ribbon and ~10 segments it is unambiguous (full HMM map matching is the
   rigorous option when segments are short and parallel).
2. **`pair_speeds`** — for each vehicle, sort by time and take **consecutive
   ping pairs**: `speed = distance / dt` (metres, EPSG:2039), attributed to the
   pair's **midpoint** segment. Filters: same `vehicle_ref`, Δt in 30–120 s,
   distance > 5 m (drop zero-distance), speed ≤ 80 km/h (drop teleports).

In [ ]:
from shapely.geometry import Point
from shapely.strtree import STRtree

_tree = STRtree(segments)

def attribute_segment(px, py):
    "Nearest-segment index for a metric point (nearest-edge snapping)."
    pt = Point(px, py)
    idx = _tree.nearest(pt)
    # STRtree.nearest returns an int index (shapely 2.x).
    return int(idx)

SPEED_CEIL_KMH = 80.0     # teleport ceiling (buses don't exceed this in-city)
DT_MIN, DT_MAX = 30.0, 120.0
DIST_MIN = 5.0            # metres; below this is GPS jitter / dwell

def pair_speeds(pings, keep_dwell=True):
    """Consecutive same-vehicle ping-pair speeds, attributed to the midpoint segment.

    pings : DataFrame with x, y, recorded_at_time, vehicle_ref (metric x,y).
    Returns a DataFrame: seg, speed_kmh, ts (pair midpoint time, UTC).
    """
    g = pings.sort_values(["vehicle_ref", "recorded_at_time"]).copy()
    by = g.groupby("vehicle_ref", sort=False)
    g["dx"] = by["x"].diff()
    g["dy"] = by["y"].diff()
    g["dt"] = by["recorded_at_time"].diff().dt.total_seconds()
    g["v0"] = by["velocity"].shift()        # previous ping's instantaneous velocity
    g["dist"] = np.hypot(g.dx, g.dy)
    g["speed_kmh"] = g.dist / g.dt * 3.6
    ok = ((g.dt >= DT_MIN) & (g.dt <= DT_MAX) & (g.dist > DIST_MIN)
          & (g.speed_kmh <= SPEED_CEIL_KMH))
    if not keep_dwell:
        # Drop pairs that begin OR end at a stop (velocity==0) — dwell out.
        ok &= (g.velocity != 0) & (g.v0 != 0)
    pr = g[ok].copy()
    # Midpoint of the pair, then nearest segment.
    pr["mx"] = pr.x - pr.dx / 2
    pr["my"] = pr.y - pr.dy / 2
    pr["seg"] = [attribute_segment(a, b) for a, b in zip(pr.mx, pr.my)]
    pr["ts"] = pr.recorded_at_time - pd.to_timedelta(pr.dt / 2, unit="s")
    return pr[["seg", "speed_kmh", "ts", "vehicle_ref"]]

pairs = pair_speeds(corr, keep_dwell=True)
print(f"Valid ping-pairs: {len(pairs):,}")
print(pairs.groupby("seg").agg(passes=("speed_kmh", "size"),
                               vehicles=("vehicle_ref", "nunique"),
                               med_kmh=("speed_kmh", "median")).round(1))

**INTERPRET (first read).** Each segment now has a count of *passes* and
*distinct vehicles*. This is the all-TLV payoff: pooling every line through the
corridor gives each segment enough passes to bin into a 15-min series. (A single
line would leave roughly half the bins empty — verified on the line-13429
slice.) Notice the median speed already separates **free-flowing** segments from
**junction-approach** ones — a preview of which will forecast well.

## 5. Per-segment speed series + the weekday/weekend floor

We bin each segment's passes to **15 min** (`AGG`), keeping only bins with
≥ `MIN_PASSES` passes (else SARIMA chases noise). The seasonal period is
`SEASON_M = 96` (daily @15min). The **historical-average floor** is built on a
**weekday vs weekend (Fri/Sat) × time-of-day** profile — *not* a full 7-day
profile, because 17 days gives only ~2-3 samples per (day, time) cell.

In [ ]:
# Config — mirrors the U3 demo's config cell.
AGG = "15min"
SEASON_M = 96                  # daily season @15min
HORIZONS = [15, 30, 60]        # minutes
STEP_MIN = int(AGG.replace("min", ""))
H_STEPS = max(HORIZONS) // STEP_MIN
HORIZON_STEPS = {h: h // STEP_MIN for h in HORIZONS}
MIN_PASSES = 2                 # minimum passes per 15-min bin to trust the mean

def segment_series(pairs, seg):
    "Pooled 15-min mean-speed series for one segment, on a regular index."
    s = pairs[pairs.seg == seg].copy()
    s["bin"] = s.ts.dt.tz_convert("Asia/Jerusalem").dt.floor(AGG)
    agg = s.groupby("bin")["speed_kmh"].agg(["mean", "size"])
    agg = agg[agg["size"] >= MIN_PASSES]["mean"]
    if len(agg) < SEASON_M * 3:        # need a few days of structure
        return None
    full = pd.date_range(agg.index.min(), agg.index.max(), freq=AGG)
    return agg.reindex(full)           # NaNs where bins were empty/thin

series = {seg: segment_series(pairs, seg) for seg in range(N_SEG)}
ok_segs = [s for s, v in series.items() if v is not None]
print(f"Segments with a usable series: {ok_segs}")
for s in ok_segs:
    v = series[s]
    print(f"  seg {s}: {v.notna().sum()} filled / {len(v)} bins "
          f"({v.notna().mean()*100:.0f}% occupied), median {v.median():.1f} km/h")

### 5a. Show, don't print — the two-week line + weekday/weekend profile

In [ ]:
import matplotlib.pyplot as plt

def weekday_weekend_profile(series_s):
    "Mean speed by time-of-day, split weekday vs weekend (Fri/Sat = dow 4,5)."
    s = series_s.dropna()
    tod = s.index.hour * 60 + s.index.minute
    wknd = s.index.dayofweek.isin([4, 5])
    prof = pd.DataFrame({"tod": tod, "wknd": wknd, "v": s.values})
    return prof.groupby(["wknd", "tod"])["v"].mean().unstack(0)

demo_seg = ok_segs[len(ok_segs) // 2]      # a representative middle segment
ss = series[demo_seg]

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
wk = ss.index.dayofweek.isin([4, 5])
ax[0].plot(ss.index, ss.values, lw=0.8, color="#1f78b4")
for d in pd.unique(ss.index[wk].normalize()):
    ax[0].axvspan(d, d + pd.Timedelta(days=1), color="#ffe0b2", alpha=0.5, zorder=0)
ax[0].set(title=f"Segment {demo_seg}: 15-min mean speed (weekends shaded)",
          ylabel="km/h")

prof = weekday_weekend_profile(ss)
for col, lab, c in [(False, "weekday", "#1f78b4"), (True, "weekend (Fri/Sat)", "#e31a1c")]:
    if col in prof.columns:
        ax[1].plot(prof.index / 60, prof[col], label=lab, color=c, lw=2)
ax[1].set(title=f"Segment {demo_seg}: weekday vs weekend daily profile",
          xlabel="hour of day", ylabel="km/h", xticks=range(0, 25, 4))
ax[1].legend()
plt.tight_layout(); plt.show()

## 6. Per-segment forecast — ADF, fixed-order SARIMA, the breakeven horizon

For each usable segment we run the demo's workflow:
- **ADF** on the (gap-filled) series — speed is bounded and mean-reverting, so we
  expect to reject the unit-root null;
- the fixed-order **SARIMA `(1,1,1)(1,1,1)_96`** (orders read off the demo's
  ACF/PACF — no AutoARIMA search, so it is fast), vs the **HistoricAverage** and
  **SeasonalNaive** floors;
- a **walk-forward** `cross_validation` with `input_size` capped to a week;
- the **breakeven horizon**: the first horizon where SARIMA's RMSE rises to meet
  the historical-average floor.

We wrap it in a helper and run it over every segment.

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsforecast import StatsForecast
from statsforecast.models import ARIMA, HistoricAverage, SeasonalNaive

N_WINDOWS = 6
INPUT_DAYS = 7
INPUT_SIZE = SEASON_M * INPUT_DAYS

def fit_breakeven(series_s):
    """Run the demo's breakeven workflow on one segment's series.

    Returns dict: adf_p, scores_wide (RMSE by horizon), breakeven (min or None).
    Gap-fills internal NaNs by time interpolation (statsforecast needs a regular
    series); leading/trailing NaNs dropped.
    """
    y = series_s.interpolate("time").dropna()
    if len(y) < INPUT_SIZE + H_STEPS * N_WINDOWS:
        return None
    adf_p = adfuller(y, autolag="AIC")[1]

    sf_df = (y.rename("y").reset_index()
             .rename(columns={"index": "ds"}))
    sf_df.columns = ["ds", "y"]
    sf_df["unique_id"] = "seg"
    sf_df = sf_df[["unique_id", "ds", "y"]]

    sf = StatsForecast(
        models=[ARIMA(order=(1, 1, 1), seasonal_order=(1, 1, 1), season_length=SEASON_M),
                HistoricAverage(),
                SeasonalNaive(season_length=SEASON_M)],
        freq=AGG, n_jobs=1)
    cv = sf.cross_validation(df=sf_df, h=H_STEPS, step_size=H_STEPS,
                             n_windows=N_WINDOWS, input_size=INPUT_SIZE)
    cvx = cv.reset_index() if cv.index.name else cv.copy()
    cvx["lead_min"] = ((cvx["ds"] - cvx["cutoff"]).dt.total_seconds() / 60).round().astype(int)

    cols = {"ARIMA": "SARIMA", "HistoricAverage": "HistoricAvg", "SeasonalNaive": "SeasonalNaive"}
    rows = []
    for h in HORIZONS:
        gh = cvx[cvx["lead_min"] == h]
        for raw, nice in cols.items():
            if raw in gh.columns:
                rmse = float(np.sqrt(((gh[raw] - gh["y"]) ** 2).mean()))
                rows.append({"horizon_min": h, "model": nice, "RMSE": rmse})
    sw = pd.DataFrame(rows).pivot(index="horizon_min", columns="model", values="RMSE")

    be = None
    for h in HORIZONS:
        if sw.loc[h, "SARIMA"] >= sw.loc[h, "HistoricAvg"]:
            be = h; break
    return {"adf_p": adf_p, "scores_wide": sw, "breakeven": be}

results = {}
for s in ok_segs:
    r = fit_breakeven(series[s])
    if r is not None:
        results[s] = r
        be = r["breakeven"]
        print(f"seg {s}: ADF p={r['adf_p']:.2g} · "
              f"breakeven = {be if be else '>'+str(max(HORIZONS))} min")

### 6a. The headline — breakeven by segment (show, don't print)

In [ ]:
be_min = {s: (r["breakeven"] if r["breakeven"] else max(HORIZONS) + 15)
          for s, r in results.items()}
segs_sorted = sorted(be_min)
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar([str(s) for s in segs_sorted], [be_min[s] for s in segs_sorted],
              color=["#33a02c" if be_min[s] > max(HORIZONS) else "#e31a1c" for s in segs_sorted])
ax.axhline(max(HORIZONS), color="#6a3d9a", ls="--", lw=1,
           label=f"max tested horizon ({max(HORIZONS)} min)")
ax.set(xlabel="segment (south -> north, downstream)",
       ylabel="breakeven horizon (min)",
       title="Which corridor segments stay predictable furthest ahead?")
ax.legend(); plt.tight_layout(); plt.show()
print("Green bars = SARIMA still beats the floor beyond the max tested horizon.")

### 6b. One segment in detail — error vs horizon + the fan chart

In [ ]:
# Pick the segment with the LONGEST breakeven for the detail view.
best_seg = max(be_min, key=lambda s: be_min[s])
sw = results[best_seg]["scores_wide"]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sw.index, sw["SARIMA"], "o-", color="#e31a1c", lw=2, label="SARIMA")
ax.plot(sw.index, sw["HistoricAvg"], "s--", color="#1f78b4", lw=2,
        label="historical average (weekday/weekend floor)")
if "SeasonalNaive" in sw.columns:
    ax.plot(sw.index, sw["SeasonalNaive"], "^:", color="#33a02c", lw=1.5,
            label="seasonal naive")
be = results[best_seg]["breakeven"]
if be is not None:
    ax.axvline(be, color="#6a3d9a", lw=1.5)
    ax.annotate(f"breakeven ~{be} min", xy=(be, ax.get_ylim()[1] * 0.9),
                xytext=(8, 0), textcoords="offset points",
                color="#6a3d9a", fontweight="bold")
ax.set(xlabel="forecast horizon (min ahead)", ylabel="RMSE (km/h)",
       title=f"Segment {best_seg}: how far ahead does SARIMA beat the floor?",
       xticks=HORIZONS)
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# A 6-hour fan forecast for the best segment, mirroring the demo's V5d.
y = series[best_seg].interpolate("time").dropna()
sf_df = y.rename("y").reset_index(); sf_df.columns = ["ds", "y"]; sf_df["unique_id"] = "seg"
sf_df = sf_df[["unique_id", "ds", "y"]]

H_FAN = 6 * 60 // STEP_MIN
cutoff = (y.index[-1] - pd.Timedelta(days=2)).normalize() + pd.Timedelta(hours=6)
train = y.loc[cutoff - pd.Timedelta(days=INPUT_DAYS):cutoff]
train_df = train.rename("y").reset_index(); train_df.columns = ["ds", "y"]
train_df["unique_id"] = "seg"; train_df = train_df[["unique_id", "ds", "y"]]
future_idx = pd.date_range(cutoff + pd.Timedelta(minutes=STEP_MIN), periods=H_FAN, freq=AGG)
actual = y.reindex(future_idx)

sf_fan = StatsForecast(
    models=[ARIMA(order=(1, 1, 1), seasonal_order=(1, 1, 1), season_length=SEASON_M),
            HistoricAverage(), SeasonalNaive(season_length=SEASON_M)],
    freq=AGG, n_jobs=1)
fan = sf_fan.forecast(df=train_df, h=H_FAN, level=[80, 95])
fx = fan["ds"]

fig, ax = plt.subplots(figsize=(13, 5))
ctx = train.loc[cutoff - pd.Timedelta(hours=12):]
ax.plot(ctx.index, ctx.values, color="#999", lw=1, label="history")
ax.plot(actual.index, actual.values, color="#1f78b4", lw=2.2, label="actual")
ax.plot(fx, fan["ARIMA"], color="#e31a1c", lw=2, label="SARIMA")
ax.fill_between(fx, fan["ARIMA-lo-95"], fan["ARIMA-hi-95"], color="#e31a1c", alpha=0.12)
ax.fill_between(fx, fan["ARIMA-lo-80"], fan["ARIMA-hi-80"], color="#e31a1c", alpha=0.22)
ax.plot(fx, fan["HistoricAverage"], color="#33a02c", ls="--", lw=1.6, label="historical avg")
ax.axvline(cutoff, color="#000", lw=1)
ax.set(xlabel="time", ylabel="speed (km/h)",
       title=f"Segment {best_seg}: 6-h SARIMA forecast with 80/95% prediction intervals")
ax.legend(fontsize=8, ncol=2); plt.tight_layout(); plt.show()

**INTERPRET.** Read the breakeven-by-segment bar chart back in transit
terms: the **free-flowing mid-block** segments (high, steady median speed) hold a
forecast edge furthest out, because their daily cycle is regular and SARIMA's
short-memory terms add little the floor doesn't already have *but* its trend
terms help near transitions. The **junction-approach** segments collapse early —
their speed is dominated by signal timing and queue spillback, which a daily
seasonal model cannot see. That contrast is the whole point: *the same corridor
contains both easy and impossible segments, and you can say which is which.*

## 7. INTERPRET on the map — segments coloured by breakeven

In [ ]:
import folium, branca.colormap as bcm

ctr = inv.transform((bx0 + bx1) / 2, (by0 + by1) / 2)
m = base_map([ctr[1], ctr[0]], zoom=15)
vmax = max(be_min.values())
cmap = bcm.linear.RdYlGn_09.scale(min(be_min.values()), vmax)
cmap.caption = "breakeven horizon (min) — green = predictable further ahead"
for s in segs_sorted:
    seg = segments[s]
    latlon = [(inv.transform(x, y)[1], inv.transform(x, y)[0]) for x, y in seg.coords]
    folium.PolyLine(latlon, color=cmap(be_min[s]), weight=6, opacity=0.9,
                    tooltip=f"seg {s}: breakeven {be_min[s]} min, "
                            f"median {series[s].median():.0f} km/h").add_to(m)
m.add_child(cmap); folium.LayerControl().add_to(m)
m

## 8. EXTEND (a) — upstream -> downstream lead — *the surprise (the U5 seed)*

Order the segments along travel and ask: does an **upstream** slowdown *lead* the
**downstream** one? We take an adjacent pair and compute the lagged
cross-correlation `corr(upstream_{t-k}, downstream_t)` over a range of lags. A
peak at lag > 0 means congestion **propagates** — exactly the spatial signal U5's
message-passing graph nets exploit.

In [ ]:
def lagged_xcorr(up, dn, max_lag=8):
    "corr(up shifted forward by k, dn) for k in 0..max_lag (k in 15-min bins)."
    a = up.interpolate("time"); b = dn.interpolate("time")
    idx = a.index.intersection(b.index)
    a, b = a.reindex(idx), b.reindex(idx)
    return {k: float(a.shift(k).corr(b)) for k in range(max_lag + 1)}

# Find an adjacent usable pair (downstream index = upstream + 1).
pair = next(((s, s + 1) for s in ok_segs if s + 1 in ok_segs), None)
if pair:
    up_s, dn_s = pair
    xc = lagged_xcorr(series[up_s], series[dn_s])
    best_lag = max(xc, key=xc.get)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar([k * STEP_MIN for k in xc], list(xc.values()), color="#1f78b4", width=8)
    ax.axvline(best_lag * STEP_MIN, color="#e31a1c", lw=2,
               label=f"peak lead = {best_lag * STEP_MIN} min")
    ax.set(xlabel="upstream lead (min)", ylabel="cross-correlation",
           title=f"Does seg {up_s} (upstream) lead seg {dn_s} (downstream)?")
    ax.legend(); plt.tight_layout(); plt.show()
    print(f"Peak lagged cross-correlation at +{best_lag * STEP_MIN} min "
          f"(corr={xc[best_lag]:.2f}).")
else:
    print("No adjacent usable segment pair — pick a denser corridor.")

**INTERPRET + the hypothesis.** If the peak sits at a positive lead, the
upstream segment's slowdown arrives at the downstream segment minutes later — the
slow patch *moves against the traffic*. The EXTEND hypothesis (stated, not built
here): adding the **upstream segment's recent speed as an exogenous regressor**
should push the downstream segment's breakeven *out* by roughly the lead time,
because at short horizons the future is already visible upstream. That is the
precise motivation for U5: ten independent SARIMAs throw this signal away; a
graph model passes it along the edge.

## 9. EXTEND (b) — segmentation A/B: junction-to-junction vs fixed 150 m

We re-cut the **same corridor spine** into uniform 150 m chunks and re-run the
breakevens. Does predictability survive the re-cut, or is it partly an artifact
of where we drew the segment boundaries?

In [ ]:
def cut_fixed(line, step=150.0):
    from shapely.ops import substring
    L = line.length; n = max(1, int(round(L / step)))
    return [substring(line, i * L / n, (i + 1) * L / n) for i in range(n)]

segments_B = cut_fixed(spine_line, 150.0)
_treeB = STRtree(segments_B)
def attr_B(px, py):
    return int(_treeB.nearest(Point(px, py)))

# Re-attribute the corridor pings to the B-segmentation by pair midpoint.
g = corr.sort_values(["vehicle_ref", "recorded_at_time"]).copy()
by = g.groupby("vehicle_ref", sort=False)
g["dx"] = by["x"].diff(); g["dy"] = by["y"].diff()
g["dt"] = by["recorded_at_time"].diff().dt.total_seconds()
g["dist"] = np.hypot(g.dx, g.dy); g["speed_kmh"] = g.dist / g.dt * 3.6
ok = (g.dt.between(DT_MIN, DT_MAX)) & (g.dist > DIST_MIN) & (g.speed_kmh <= SPEED_CEIL_KMH)
gB = g[ok].copy(); gB["mx"] = gB.x - gB.dx / 2; gB["my"] = gB.y - gB.dy / 2
gB["seg"] = [attr_B(a, b) for a, b in zip(gB.mx, gB.my)]
gB["ts"] = gB.recorded_at_time - pd.to_timedelta(gB.dt / 2, unit="s")
pairsB = gB[["seg", "speed_kmh", "ts", "vehicle_ref"]]

seriesB = {s: segment_series(pairsB, s) for s in range(len(segments_B))}
beB = {}
for s, v in seriesB.items():
    if v is None: continue
    r = fit_breakeven(v)
    if r: beB[s] = r["breakeven"] if r["breakeven"] else max(HORIZONS) + 15
print(f"Junction-to-junction (A): {len(be_min)} segs, breakevens {sorted(be_min.values())}")
print(f"Fixed 150 m (B):          {len(beB)} segs, breakevens {sorted(beB.values())}")

**INTERPRET.** Compare the two breakeven distributions. If the fixed-150 m
cut produces *more* short-breakeven segments, it is because uniform chunks
straddle junctions and mix a free block with a queue — blurring a signal the
junction-to-junction cut isolated. If it produces *fewer*, the equal lengths gave
every segment enough passes where per-edge segments starved the short ones. Which
to ship depends on the question: junction-to-junction for *interpretability*
(each segment is a real block), fixed-length for *uniform data density*. Either
way, the lesson is sharp: **predictability is partly a property of how you cut,
not only of the road** — own the cut.

## 10. EXTEND (c) — dwell in or out (the `velocity == 0` decision)

Rebuild one segment's series **excluding** stopped pings (dwell out) and compare
its distribution, ADF, and breakeven to the dwell-in version. Does dropping the
stops make the segment *more* forecastable, or just hide the congestion we wanted
to predict?

In [ ]:
seg_c = best_seg
pairs_nodwell = pair_speeds(corr, keep_dwell=False)
s_in = series[seg_c]
s_out = segment_series(pairs_nodwell, seg_c)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(s_in.dropna(), bins=30, alpha=0.6, label="dwell IN", color="#1f78b4")
if s_out is not None:
    ax[0].hist(s_out.dropna(), bins=30, alpha=0.6, label="dwell OUT", color="#e31a1c")
ax[0].set(title=f"Segment {seg_c}: speed distribution", xlabel="km/h"); ax[0].legend()

r_in = results[seg_c]
print(f"dwell IN : ADF p={r_in['adf_p']:.2g} · breakeven {r_in['breakeven']}")
if s_out is not None and s_out.notna().sum() > SEASON_M * 3:
    r_out = fit_breakeven(s_out)
    if r_out:
        print(f"dwell OUT: ADF p={r_out['adf_p']:.2g} · breakeven {r_out['breakeven']}")
        ax[1].plot(r_in["scores_wide"].index, r_in["scores_wide"]["SARIMA"], "o-",
                   label="dwell IN", color="#1f78b4")
        ax[1].plot(r_out["scores_wide"].index, r_out["scores_wide"]["SARIMA"], "s-",
                   label="dwell OUT", color="#e31a1c")
        ax[1].set(title="SARIMA RMSE vs horizon", xlabel="min ahead", ylabel="RMSE")
        ax[1].legend()
plt.tight_layout(); plt.show()

**INTERPRET.** Dropping `velocity == 0` pings shifts the distribution
right (no stopped samples) and usually makes ADF "nicer" (less variance). But the
stops *are* the congestion at a junction-approach segment — removing them can
**hide** the very signal that made the forecast useful. Unlike METR-LA's
"0 = missing", here 0 is a real state. The defensible call: keep dwell for
junction segments (you want to predict the queue), consider dropping it for
mid-block free-flow where stops are GPS noise.

## 11. EXTEND (d) — when is per-segment SARIMA the *wrong* tool? *(meta)*

Three conditions where ten independent univariate SARIMAs are the wrong choice,
and what later units supply:

1. **A starved segment** — too few passes per bin for *any* univariate fit; the
   floor wins at every horizon. Fix: pool more (wider segment, longer window) or
   borrow strength from neighbours -> a **spatial** model.
2. **An anomaly in the window** — Feb 2023 may contain a strike or a storm; a
   daily-seasonal model cannot see a one-off. Fix: U3's own outlier/intervention
   reasoning, or exogenous regressors.
3. **The spatial signal we found in (a)** — the upstream lead means a *graph*
   model that passes messages along the corridor (U5) should beat ten
   independent SARIMAs at short horizons. This is the bridge to Unit 5.

The right read: SARIMA is the **honest floor-beater**, not the ceiling. Knowing
*where* it stops being right is the unit's real deliverable.

## 12. Where to go next + decision-log mapping

- **Baseline** -> log entries: corridor pick + **named segmentation strategy**;
  snap + ping-pair speed filters; per-segment breakeven table; the
  interpret-the-contrast paragraph.
- **EXTEND (a)** the upstream lead -> the U5 seed (the surprise).
- **EXTEND (b)** segmentation A/B -> predictability is partly an artifact of the
  cut.
- **EXTEND (c)** dwell in/out -> the `velocity==0` decision.
- **EXTEND (d)** wrong-class -> names what U5 (graph message passing) fixes.
- **Homework** -> sum segment crossing-times into a **corridor travel-time
  budget** and compare its breakeven to the parts.

**Unit 5 next:** replace the ten independent SARIMAs with one **spatio-temporal
graph network** over the segment graph — the upstream lead from (a) becomes an
edge the model learns to use.

## Appendix — provenance (NOT on the runtime path)

`tlv_all.parquet` and `tlv_2039_backup.graphml.gz` are the same artifacts U2
produced (see `instructor-shared/local-tasks/unit2-cut-tlv-data.md`). The cell
below documents the regeneration recipe; it is inert (`REGENERATE = False`).

In [ ]:
REGENERATE = False   # documentation only — the runtime path uses the hosted slice.
if REGENERATE:
    # From the raw archive (1.22 GB ZIP -> exports/vehicle_locations.parquet, 730 MB):
    #   1. read raw parquet; keep the 7 columns;
    #   2. clip to the TLV bbox lat[32.00,32.15] lon[34.74,34.88];
    #   3. write tlv_all.parquet (~48 MB) and upload to Drive (DATA_ID);
    #   4. build the OSM drive graph for the bbox, project to EPSG:2039,
    #      gzip -> tlv_2039_backup.graphml.gz, upload (BACKUP_GRAPH_ID).
    pass